# Ultra-Simple RAG (Retrieval-Augmented Generation)

This notebook demonstrates how to build a simple but effective RAG system that combines document retrieval with language generation. RAG systems enhance language models by providing them with relevant context from a knowledge base, enabling more accurate and informed responses.

## Overview
- Load and process a document (Paul Graham essay)
- Split text into manageable chunks
- Create embeddings for semantic search
- Build a FAISS vector index for fast retrieval
- Implement question-answering with retrieved context

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

## Setup and Configuration

### Configure IPython Display
Set up IPython to display all expressions in a cell for better debugging and exploration.

In [3]:
import requests
import numpy as np
import faiss
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

W0705 09:33:00.742000 61626 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


### Import Required Libraries
Import all necessary libraries for building our RAG system:
- **requests**: For fetching documents from URLs
- **numpy & faiss**: For vector operations and similarity search
- **torch**: PyTorch for deep learning operations
- **transformers**: Hugging Face models for text generation
- **langchain**: Text splitting utilities
- **sentence_transformers**: For creating high-quality embeddings

In [4]:
response = requests.get('https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt')
text = response.text

## Document Loading and Processing

### Load Source Document
Fetch Paul Graham's essay from the web - this will serve as our knowledge base.
The essay contains rich content about startups, programming, and entrepreneurship.

In [5]:
print(text[700: 1000])

e a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t


### Preview Document Content
Display a sample of the loaded text to understand the content structure and style.

In [26]:
CHUNK_SIZE = 1500    # Size of each chunk
CHUNK_OVERLAP = 100

### Configure Text Chunking Parameters
Set up chunking parameters for optimal retrieval performance:
- **CHUNK_SIZE (1400)**: Large enough to contain meaningful context
- **CHUNK_OVERLAP (100)**: Ensures continuity between adjacent chunks

In [27]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ",],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP

)

### Initialize Text Splitter
Create a RecursiveCharacterTextSplitter that intelligently splits text:
- Tries paragraph breaks first (\n\n)
- Falls back to sentence breaks (\n)
- Finally splits on spaces if needed
- Maintains semantic coherence in chunks

### Split Document into Chunks
Break the document into overlapping chunks for better retrieval granularity.
Display the total number of chunks created for the knowledge base.

In [28]:
# chunks = [text[i - CHUNK_OVERLAP:i + CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]
chunks = text_splitter.split_text(text)
print(len(chunks))

61


### Sample Chunk Preview
Display random chunks to verify the splitting quality and content distribution.
This helps ensure chunks contain meaningful, coherent information.

In [29]:
for sample_text in random.sample(chunks, 3):  # Display 3 random chunks
        print(sample_text[:500])  # Print the first 200 characters of each chunk
        print("")

We did a lot of things right by accident like that. For example, we did what's now called "doing things that don't scale," although at the time we would have described it as "being so lame that we're driven to the most desperate measures to get users." The most common of which was building stores for them. This seemed particularly humiliating, since the whole raison d'etre of our software was that people could use it to make their own stores. But anything to get users.

We learned a lot more abo

I liked painting still lives because I was curious about what I was seeing. In everyday life, we aren't consciously aware of much we're seeing. Most visual perception is handled by low-level processes that merely tell your brain "that's a water droplet" without telling you details like where the lightest and darkest points are, or "that's a bush" without telling you the shape and position of every leaf. This is a feature of brains, not a bug. In everyday life it would be distracting to notice 

## Embedding and Vector Index Creation

### Create Document Embeddings
Use SentenceTransformer to encode all chunks into dense vector representations:
- **all-MiniLM-L6-v2**: Efficient model with good semantic understanding
- **normalize=True**: Ensures consistent vector magnitudes for better similarity calculation
- Check max_seq_length to ensure chunks fit within model limits

In [30]:
# use sentence transformer model with large max_seq_length
# you can use any model, but this one is known to work well with long texts
embeddings_model = SentenceTransformer('all-MiniLM-L6-v2')
print(embeddings_model.max_seq_length)
embeddings = np.array([embeddings_model.encode(chunk, normalize=True) for chunk in chunks])

256


### Build FAISS Vector Index
Create a FAISS index for fast similarity search:
- **IndexFlatL2**: Uses L2 (Euclidean) distance for similarity
- **dimension**: Matches the embedding model's output dimension
- Add all document embeddings to enable semantic search

In [31]:
# lets create a FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension, )
index.add(embeddings)

## Retrieval and Testing

### Test Retrieval System
Query the vector index to find relevant document chunks:
- Encode the question using the same embedding model
- Search for top-k most similar chunks
- Display results with similarity distances for evaluation

In [32]:
# lets test the index
question = "Did Yahoo stock go up?"
question_embeddings = np.array([embeddings_model.encode(question, normalize=True)])
distances, indices = index.search(question_embeddings, k=5, )

for i, idx in enumerate(indices[0]):
    print(f"Chunk {i+1} (Distance: {distances[0][i]}):")
    print(chunks[idx][:200])  # Print the first 200 characters of each chunk
    print()  # Add a newline for better readability

Chunk 1 (Distance: 1.0820555686950684):
It was a huge relief when Yahoo bought us. In principle our Viaweb stock was valuable. It was a share in a business that was profitable and growing rapidly. But it didn't feel very valuable to me; I h

Chunk 2 (Distance: 1.2395577430725098):
Another thing I didn't get at the time is that growth rate is the ultimate test of a startup. Our growth rate was fine. We had about 70 stores at the end of 1996 and about 500 at the end of 1997. I mi

Chunk 3 (Distance: 1.3802547454833984):
Yahoo had given us a lot of options when they bought us. At the time I thought Yahoo was so overvalued that they'd never be worth anything, but to my astonishment the stock went up 5x in the next year

Chunk 4 (Distance: 1.3998780250549316):
So I gave this talk, in the course of which I told them that the best sources of seed funding were successful startup founders, because then they'd be sources of advice too. Whereupon it seemed they w

Chunk 5 (Distance: 1.47162401676

### Extract Retrieved Context
Collect the most relevant chunks that will provide context for answer generation.
Preview the retrieved content to verify relevance to the query.

In [33]:
retrieved_chunk = [chunks[i] for i in indices.tolist()[0]]
print("Retrieved Chunks:")
for chunk in retrieved_chunk:
    print(chunk[:200])  # Print the first 200 characters of each chunk
    print()  # Add a newline for better readability

Retrieved Chunks:
It was a huge relief when Yahoo bought us. In principle our Viaweb stock was valuable. It was a share in a business that was profitable and growing rapidly. But it didn't feel very valuable to me; I h

Another thing I didn't get at the time is that growth rate is the ultimate test of a startup. Our growth rate was fine. We had about 70 stores at the end of 1996 and about 500 at the end of 1997. I mi

Yahoo had given us a lot of options when they bought us. At the time I thought Yahoo was so overvalued that they'd never be worth anything, but to my astonishment the stock went up 5x in the next year

So I gave this talk, in the course of which I told them that the best sources of seed funding were successful startup founders, because then they'd be sources of advice too. Whereupon it seemed they w

I had not originally intended YC to be a full-time job. I was going to do three things: hack, write essays, and work on YC. As YC grew, and I grew more excited about it, it s

## Answer Generation

### Create RAG Prompt
Construct a structured prompt that includes:
- Retrieved context from similar document chunks
- Clear instructions for the language model
- The original user question
- Request for a context-based answer

In [34]:
prompt = f"""
Context information is below.
---------------------
{"\n".join([": ".join([str(i), c]) for i, c in enumerate(retrieved_chunk)])}
---------------------
Given the context information, answer the query.
Query: {question}
Answer:
"""

### Load Language Model
Initialize a capable language model for answer generation:
- **Gemma-2-2b-it**: Instruction-tuned model optimized for chat/QA
- **bfloat16**: Reduces memory usage while maintaining quality
- **device_map="auto"**: Automatically distributes model across available hardware

In [35]:
model_name = "google/gemma-3-1b-it"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

### Check Model Memory Usage
Monitor the model's memory footprint to ensure it fits within available resources.

In [36]:
model.get_memory_footprint()/1024**3  # in GB

1.862433673813939

### Generate Final Answer
Process the RAG prompt through the language model:
- Apply chat template for proper formatting
- Generate response with controlled randomness (temperature=0.2)
- Extract only the answer portion (excluding the input prompt)
- Display the final context-aware response

In [37]:
input_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.2, top_p=0.95)
# only include the answer part of the output
# Extract only the newly generated tokens (exclude the input prompt)
answer_tokens = outputs[0][len(inputs.input_ids[0]):]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True)
print("Answer:", answer)

Answer: Yes, Yahoo stock went up 5x in the next year after being bought by Yahoo.<end_of_turn>
